# Random Functions

A **random function** is a distribution over functions.  Where a standard
distribution like `Normal(loc=0, scale=1)` produces *scalar* samples, a
random function produces *callable* samples — entire functions that can be
evaluated at arbitrary inputs.

ProbPipe represents this concept with the `RandomFunction[X, Y]` class, a
subclass of `Distribution[T]` where `T = Callable`.  The primary interface
is **calling** the object: given inputs, it returns a distribution over
outputs (the finite-dimensional marginal at those inputs).

This notebook walks through the full random-function hierarchy:

1. **`RandomFunction[X, Y]`** — generic base (distribution over functions)
2. **`ArrayRandomFunction`** — specialisation for array-valued inputs/outputs with TFP-style shape semantics
3. **`GaussianRandomFunction`** — guarantees Gaussian predictive distributions
4. **`LinearBasisFunction`** — concrete model: `f(x) = a + Φ(x) @ w`,  `w ~ N(m, C)`
5. **`LinearOutputTransform`** — wraps a Gaussian random function through a linear map
6. **`EmulatorMixin`** — mixin for training-data-aware models

We also demonstrate how random-function outputs interact with standard
ProbPipe features: **expectations**, **provenance tracking**, and
**WorkflowFunction broadcasting**.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from probpipe import (
    Distribution,
    RandomFunction,
    ArrayRandomFunction,
    GaussianRandomFunction,
    LinearBasisFunction,
    LinearOutputTransform,
    EmulatorMixin,
    MultivariateNormal,
    Normal,
    Provenance,
    provenance_ancestors,
    set_return_approx_dist,
)

key = jax.random.PRNGKey(0)

# For cleaner output in this notebook, return plain arrays from expectations.
set_return_approx_dist(False)

## 1. The Type Hierarchy

Every random function is a `Distribution`.  The hierarchy is:

```
Distribution[T]
  └── RandomFunction[X, Y]           # T = Callable; __call__ is abstract
        └── ArrayRandomFunction       # X = Array, Y = Array; shape contract
              └── GaussianRandomFunction  # Gaussian predictive distributions
                    ├── LinearBasisFunction
                    └── LinearOutputTransform
```

`RandomFunction` sits alongside `ArrayDistribution` and `JointDistribution`
in the distribution hierarchy — it is a peer, not a replacement.

## 2. Shape Semantics (`ArrayRandomFunction`)

For array-valued random functions, calling the object with input `X` of
shape `(*extra_batch, n, *input_shape)` returns a distribution whose
`batch_shape` and `event_shape` depend on two boolean flags:

| `joint_inputs` | `joint_outputs` | `event_shape`         | `batch_shape`                    |
|:--------------:|:---------------:|:---------------------:|:--------------------------------:|
| False          | False           | `()`                  | `(*extra_batch, n, *output_shape)` |
| True           | False           | `(n,)`                | `(*extra_batch, *output_shape)`  |
| False          | True            | `(*output_shape,)`    | `(*extra_batch, n)`              |
| True           | True            | `(n, *output_shape)`  | `(*extra_batch,)`                |

In all modes the total shape of a sample is invariant:
`(*sample_shape, *extra_batch, n, *output_shape)`.  The flags only control
which axes are jointly modelled (event) versus independent (batch).

## 3. Building a Custom `GaussianRandomFunction`

The simplest way to create a random function is to subclass
`GaussianRandomFunction` and implement `predict_mean`, `predict_variance`,
and (optionally) `predict_covariance`.

Below we build a zero-mean GP prior with a squared-exponential (RBF) kernel.
Because the GP provides a full covariance matrix across input points, we set
`supports_joint_inputs = True`.

In [ ]:
def rbf_kernel(X1, X2, lengthscale=1.0, variance=1.0):
    """Squared-exponential (RBF) kernel."""
    sq_dist = jnp.sum((X1[:, None, :] - X2[None, :, :]) ** 2, axis=-1)
    return variance * jnp.exp(-0.5 * sq_dist / lengthscale ** 2)


class ToyGP(GaussianRandomFunction):
    """Zero-mean GP with an RBF kernel."""

    supports_joint_inputs = True

    def __init__(self, lengthscale=1.0, variance=1.0, noise=0.01):
        super().__init__(input_shape=(1,), output_shape=())
        self._ls = lengthscale
        self._var = variance
        self._noise = noise

    def predict_mean(self, X):
        extra_batch, n = self._parse_X(X)
        return jnp.zeros((*extra_batch, n))

    def predict_variance(self, X):
        extra_batch, n = self._parse_X(X)
        return jnp.full((*extra_batch, n), self._var)

    def predict_covariance(self, X, *, joint_inputs=False, joint_outputs=False):
        if not joint_inputs:
            raise NotImplementedError("Use predict_variance for marginals.")
        K = rbf_kernel(X, X, self._ls, self._var)
        return K + self._noise * jnp.eye(K.shape[-1])


gp = ToyGP(lengthscale=0.3, variance=1.0, noise=1e-4)
print(gp)
print(f"  isinstance(gp, Distribution): {isinstance(gp, Distribution)}")
print(f"  isinstance(gp, RandomFunction): {isinstance(gp, RandomFunction)}")

### 3.1 Marginal predictions (default mode)

Calling the GP with `n` input points returns a batch of `n` independent
`Normal` distributions — one per input point.

In [ ]:
X_test = jnp.linspace(-2, 2, 50).reshape(-1, 1)

# Default: joint_inputs=False, joint_outputs=False
marginal = gp(X_test)

print(f"Type:        {type(marginal).__name__}")
print(f"batch_shape: {marginal.batch_shape}")
print(f"event_shape: {marginal.event_shape}")

# Plot marginal mean +/- 2 std
mean = marginal.mean()
std = jnp.sqrt(marginal.variance())

fig, ax = plt.subplots(figsize=(8, 3))
ax.fill_between(X_test[:, 0], mean - 2 * std, mean + 2 * std, alpha=0.3, label="mean +/- 2 std")
ax.plot(X_test[:, 0], mean, "k-", lw=1.5, label="mean")
ax.set(xlabel="x", ylabel="f(x)", title="GP Marginal Predictions")
ax.legend()
plt.tight_layout()

### 3.2 Joint predictions (`joint_inputs=True`)

With `joint_inputs=True`, the GP returns a single `MultivariateNormal`
whose event dimension spans all `n` input points.  This lets us draw
*correlated* samples (smooth function trajectories) rather than independent
pointwise samples.

In [ ]:
joint = gp(X_test, joint_inputs=True)

print(f"Type:        {type(joint).__name__}")
print(f"batch_shape: {joint.batch_shape}")
print(f"event_shape: {joint.event_shape}")

# Draw correlated samples (smooth trajectories)
k1, k2 = jax.random.split(key)
trajectories = joint.sample(k1, sample_shape=(5,))  # (5, 50)

fig, ax = plt.subplots(figsize=(8, 3))
for i in range(5):
    ax.plot(X_test[:, 0], trajectories[i], alpha=0.7, lw=1)
ax.fill_between(X_test[:, 0], mean - 2 * std, mean + 2 * std, alpha=0.15, color="gray")
ax.set(xlabel="x", ylabel="f(x)", title="GP Samples (joint_inputs=True)")
plt.tight_layout()

### 3.3 Expectations on predictive distributions

The distribution returned by a random function is a standard ProbPipe
distribution.  All the usual operations work — `mean()`, `variance()`,
`sample()`, `expectation()`, `log_prob()`.

In [ ]:
# Evaluate at a single input point
x_star = jnp.array([[0.5]])
pred = gp(x_star)

print(f"Predictive distribution at x=0.5: {type(pred).__name__}")
print(f"  mean:     {float(pred.mean()):.4f}")
print(f"  variance: {float(pred.variance()):.4f}")

# Compute E[sin(f(x*))] under the predictive distribution
E_sin = pred.expectation(jnp.sin, key=k2, num_evaluations=5000, return_dist=False)
print(f"  E[sin(f)]: {float(E_sin):.4f}")

# Log-density at the mean
print(f"  log p(0): {float(pred.log_prob(jnp.array(0.0))):.4f}")

## 4. `LinearBasisFunction`

`LinearBasisFunction` is a concrete `GaussianRandomFunction` implementing
the model:

$$f(x) = a + \Phi(x)\, w, \qquad w \sim \mathcal{N}(m, C)$$

where $\Phi(x)$ is a user-supplied **feature map** and $w$ is a Gaussian
weight vector.  Because the model is linear in $w$, the predictive
distribution at any finite set of inputs is Gaussian with analytically
available mean, variance, and covariance.

This class also supports **function sampling**: since a function realization
is determined by a finite weight vector, `sample()` returns a callable that
evaluates the linear model at arbitrary inputs using a single weight draw.

In [ ]:
def polynomial_features(X, degree=5):
    """Feature map: [1, x, x^2, ..., x^degree] for scalar input."""
    x = X[..., 0]  # (*extra_batch, n)
    return jnp.stack([x ** k for k in range(degree + 1)], axis=-1)


# Weight distribution: 6 basis functions with small prior uncertainty
d_w = 6
w_mean = jnp.zeros(d_w)
w_cov = 0.5 * jnp.eye(d_w)
weights = MultivariateNormal(loc=w_mean, cov=w_cov)

lbf = LinearBasisFunction(
    feature_map=polynomial_features,
    weights=weights,
    input_shape=(1,),
)

print(lbf)
print(f"  supports_joint_inputs:  {lbf.supports_joint_inputs}")
print(f"  supports_joint_outputs: {lbf.supports_joint_outputs}")

### 4.1 Predictive distributions and sampling

In [ ]:
X_plot = jnp.linspace(-1.5, 1.5, 100).reshape(-1, 1)

# Marginal predictions
pred = lbf(X_plot)
mu = pred.mean()
sigma = jnp.sqrt(pred.variance())

# Joint predictions for correlated samples
joint_pred = lbf(X_plot, joint_inputs=True)
k3, k4 = jax.random.split(k2)
joint_samples = joint_pred.sample(k3, sample_shape=(8,))  # (8, 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

ax = axes[0]
ax.fill_between(X_plot[:, 0], mu - 2*sigma, mu + 2*sigma, alpha=0.3)
ax.plot(X_plot[:, 0], mu, "k-", lw=1.5)
ax.set(xlabel="x", ylabel="f(x)", title="Marginal: mean +/- 2 std")

ax = axes[1]
for i in range(8):
    ax.plot(X_plot[:, 0], joint_samples[i], alpha=0.6, lw=1)
ax.fill_between(X_plot[:, 0], mu - 2*sigma, mu + 2*sigma, alpha=0.1, color="gray")
ax.set(xlabel="x", title="Joint samples (correlated trajectories)")
plt.tight_layout()

### 4.2 Function sampling (`sample`)

Because `LinearBasisFunction` is a finite-dimensional model (parameterized
by a weight vector), it supports **function sampling** — `sample()` returns
a *callable* that evaluates the sampled function at arbitrary inputs.

This is fundamentally different from sampling the predictive distribution
at fixed inputs: here we get a single coherent function that can be queried
at any new point.

In [ ]:
# Draw a single function realization
k5, k6 = jax.random.split(k4)
f_single = lbf.sample(k5)

print(f"Type of sample: {type(f_single)}")
print(f"Callable?       {callable(f_single)}")

# Evaluate the sampled function at different input sets
y1 = f_single(X_plot)
y2 = f_single(jnp.array([[0.0], [1.0], [-1.0]]))

print(f"\nEvaluated at 100 points: shape {y1.shape}")
print(f"Evaluated at 3 points:   shape {y2.shape}")

# The same function evaluated twice at the same input gives the same output
y1_again = f_single(X_plot)
print(f"\nConsistent? {jnp.allclose(y1, y1_again)}")

# Draw multiple function realizations at once
f_batch = lbf.sample(k6, sample_shape=(5,))
y_batch = f_batch(X_plot)  # (5, 100)
print(f"\nBatched evaluation shape: {y_batch.shape}")

fig, ax = plt.subplots(figsize=(8, 3))
for i in range(5):
    ax.plot(X_plot[:, 0], y_batch[i], alpha=0.6, lw=1, label=f"sample {i}")
ax.plot(X_plot[:, 0], y1, "k--", lw=2, label="single sample")
ax.fill_between(X_plot[:, 0], mu - 2*sigma, mu + 2*sigma, alpha=0.1, color="gray")
ax.set(xlabel="x", ylabel="f(x)", title="Function Sampling (callable realizations)")
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()

## 5. `LinearOutputTransform`

`LinearOutputTransform` creates a new Gaussian random function by linearly
transforming the outputs of an existing one:

$$f(x) = a + \Phi\, g(x)$$

where $g(x)$ is a `GaussianRandomFunction` with 1-D output shape `(d_w,)`,
and $\Phi$ is a fixed matrix mapping from $g$'s output space to the new
output space.

This is useful when you have a latent Gaussian model (e.g. over basis
weights) and want to project it into a different observation space.

In [ ]:
# Build a base function that outputs a 3-D latent vector at each input
def latent_features(X):
    """Feature map: [1, x, x^2] for scalar input -> 3-D output."""
    x = X[..., 0]
    return jnp.stack([jnp.ones_like(x), x, x ** 2], axis=-1)

latent_weights = MultivariateNormal(
    loc=jnp.zeros(3),
    cov=jnp.eye(3),
)
base_fn = LinearBasisFunction(
    feature_map=latent_features,
    weights=latent_weights,
    input_shape=(1,),
    output_shape=(3,),  # 3-D latent output
)

# Project from 3-D latent space to 2-D observation space
Phi = jnp.array([[1.0, 0.5, 0.0],
                  [0.0, 0.3, 1.0]])
bias = jnp.array([0.1, -0.2])

lot = LinearOutputTransform(base_function=base_fn, phi=Phi, bias=bias)

print(lot)
print(f"  input_shape:            {lot.input_shape}")
print(f"  output_shape:           {lot.output_shape}")
print(f"  supports_joint_inputs:  {lot.supports_joint_inputs}")
print(f"  supports_joint_outputs: {lot.supports_joint_outputs}")
print(f"  isinstance(lot, GaussianRandomFunction): {isinstance(lot, GaussianRandomFunction)}")

In [ ]:
# Marginal predictions: batch of 2-D Normal distributions
X_lot = jnp.linspace(-1.5, 1.5, 60).reshape(-1, 1)
pred_lot = lot(X_lot)

print(f"Type:        {type(pred_lot).__name__}")
print(f"batch_shape: {pred_lot.batch_shape}")
print(f"event_shape: {pred_lot.event_shape}")

lot_mean = pred_lot.mean()       # (60, 2)
lot_std = jnp.sqrt(pred_lot.variance())  # (60, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for i, label in enumerate(["Output 0", "Output 1"]):
    ax = axes[i]
    ax.fill_between(
        X_lot[:, 0],
        lot_mean[:, i] - 2 * lot_std[:, i],
        lot_mean[:, i] + 2 * lot_std[:, i],
        alpha=0.3,
    )
    ax.plot(X_lot[:, 0], lot_mean[:, i], "k-", lw=1.5)
    ax.set(xlabel="x", ylabel="f(x)", title=f"LinearOutputTransform — {label}")
plt.tight_layout()

## 6. `EmulatorMixin` — Training-Data-Aware Models

`EmulatorMixin` is an orthogonal mixin that adds a training-data interface
to any random function.  It provides four optional hooks:

- `fit(X, Y)` — train on data
- `update(X_new, Y_new)` — incorporate new observations
- `training_inputs` — property returning stored inputs
- `training_responses` — property returning stored responses

All raise `NotImplementedError` by default; subclasses implement whichever
ones apply.  This design means "emulator" behaviour is *composable* — you
can mix it into any level of the hierarchy.

In [ ]:
class FittedPolynomial(LinearBasisFunction, EmulatorMixin):
    """A LinearBasisFunction that can be fit to data via least-squares."""

    def __init__(self, degree=3):
        self._degree = degree
        d_w = degree + 1
        # Start with a vague prior
        super().__init__(
            feature_map=lambda X: polynomial_features(X, degree=degree),
            weights=MultivariateNormal(
                loc=jnp.zeros(d_w),
                cov=100.0 * jnp.eye(d_w),
            ),
            input_shape=(1,),
        )
        self._X_train = None
        self._Y_train = None

    def fit(self, X, Y):
        """Bayesian linear regression: update weights given data."""
        X, Y = jnp.asarray(X), jnp.asarray(Y)
        self._X_train = X
        self._Y_train = Y

        phi = self._feature_map(X)  # (n, d_w)
        noise_var = 0.1

        # Posterior: w | data ~ N(m_post, C_post)
        prior_prec = jnp.linalg.inv(self._w_cov)
        C_post = jnp.linalg.inv(prior_prec + phi.T @ phi / noise_var)
        m_post = C_post @ (prior_prec @ self._w_mean + phi.T @ Y / noise_var)

        self._w_mean = m_post
        self._w_cov = C_post
        self._weights = MultivariateNormal(loc=m_post, cov=C_post)

    @property
    def training_inputs(self):
        if self._X_train is None:
            raise NotImplementedError("Not yet fit.")
        return self._X_train

    @property
    def training_responses(self):
        if self._Y_train is None:
            raise NotImplementedError("Not yet fit.")
        return self._Y_train


# Generate noisy data from a true function
true_fn = lambda x: jnp.sin(2 * x)
X_train = jnp.linspace(-1.5, 1.5, 15).reshape(-1, 1)
Y_train = true_fn(X_train[:, 0]) + 0.2 * jax.random.normal(k6, shape=(15,))

# Fit the model
model = FittedPolynomial(degree=5)
model.fit(X_train, Y_train)

print(f"isinstance(model, LinearBasisFunction): {isinstance(model, LinearBasisFunction)}")
print(f"isinstance(model, EmulatorMixin):       {isinstance(model, EmulatorMixin)}")
print(f"Training points: {model.training_inputs.shape[0]}")

# Predict
pred_fit = model(X_plot)
mu_fit = pred_fit.mean()
sigma_fit = jnp.sqrt(pred_fit.variance())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.fill_between(X_plot[:, 0], mu_fit - 2*sigma_fit, mu_fit + 2*sigma_fit, alpha=0.3, label="posterior +/- 2 std")
ax.plot(X_plot[:, 0], mu_fit, "k-", lw=1.5, label="posterior mean")
ax.plot(X_plot[:, 0], true_fn(X_plot[:, 0]), "r--", lw=1, label="true function")
ax.scatter(X_train[:, 0], Y_train, c="red", s=30, zorder=5, label="training data")
ax.set(xlabel="x", ylabel="f(x)", title="EmulatorMixin: Bayesian Polynomial Regression")
ax.legend(fontsize=8)
plt.tight_layout()

## 7. Provenance Tracking

Distributions produced by random functions carry **provenance** just like
any other ProbPipe distribution.  Provenance records *how* a distribution
was created (operation, parent distributions, metadata) and supports
ancestor tracing via `provenance_ancestors()`.

In [ ]:
# Create a named distribution and pass it through a random function workflow
x_dist = Normal(loc=0.0, scale=0.5, name="x_input")
print(f"x_dist: {x_dist}")
print(f"  source: {x_dist.source}")

# Use the fitted model to predict — the result is a standard distribution
x_query = jnp.array([[0.3]])
pred_at_query = model(x_query)

# Attach provenance manually to show how it works
pred_at_query.with_source(Provenance(
    "random_function_predict",
    parents=(),
    metadata={"model": type(model).__name__, "x": 0.3},
))

print(f"\nPrediction at x=0.3:")
print(f"  type:      {type(pred_at_query).__name__}")
print(f"  source:    {pred_at_query.source}")
print(f"  operation: {pred_at_query.source.operation}")
print(f"  metadata:  {pred_at_query.source.metadata}")

## 8. WorkflowFunction Broadcasting

`WorkflowFunction` automatically **broadcasts** over `Distribution`
arguments.  When a function expects a plain array but receives a
distribution, ProbPipe samples from that distribution and evaluates
the function at each sample.

Random function outputs are standard distributions, so they integrate
seamlessly with this broadcasting mechanism.

In [ ]:
from probpipe import WorkflowFunction

# A plain function that takes a scalar and returns a scalar
def nonlinear_transform(y: float) -> float:
    """A nonlinear function we want to propagate uncertainty through."""
    return jnp.sin(y) + 0.5 * y ** 2

# Wrap it as a WorkflowFunction
wf = WorkflowFunction(func=nonlinear_transform, n_broadcast_samples=500, seed=42)

# Get the predictive distribution at a single input point from our fitted model
pred_at_0 = model(jnp.array([[0.0]]))
print(f"Predictive distribution at x=0: {type(pred_at_0).__name__}")
print(f"  mean={float(pred_at_0.mean()):.3f}, std={float(jnp.sqrt(pred_at_0.variance())):.3f}")

# Broadcast: pass a Distribution where the function expects a float
# WorkflowFunction samples from pred_at_0 and evaluates nonlinear_transform
# at each sample, returning an EmpiricalDistribution
result = wf(y=pred_at_0)

print(f"\nResult type: {type(result).__name__}")
print(f"  mean:     {float(result.mean()):.4f}")
print(f"  variance: {float(result.variance()):.4f}")

# Provenance records the broadcasting operation
print(f"\n  source operation: {result.source.operation}")
print(f"  source metadata:  {result.source.metadata}")

In [ ]:
# Visualize: propagating uncertainty through a nonlinear function
# at multiple input locations
X_prop = jnp.linspace(-1.0, 1.0, 30).reshape(-1, 1)
pred_prop = model(X_prop)  # batch of 30 Normal distributions

# For each input, broadcast through the nonlinear transform
results = []
for i in range(X_prop.shape[0]):
    pred_i = Normal(loc=pred_prop.mean()[i], scale=jnp.sqrt(pred_prop.variance()[i]))
    r = wf(y=pred_i)
    results.append((float(r.mean()), float(r.variance())))

r_mean = jnp.array([r[0] for r in results])
r_std = jnp.sqrt(jnp.array([r[1] for r in results]))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

ax = axes[0]
ax.fill_between(X_prop[:, 0], pred_prop.mean() - 2*jnp.sqrt(pred_prop.variance()),
                pred_prop.mean() + 2*jnp.sqrt(pred_prop.variance()), alpha=0.3)
ax.plot(X_prop[:, 0], pred_prop.mean(), "k-", lw=1.5)
ax.set(xlabel="x", ylabel="f(x)", title="Model Prediction (input)")

ax = axes[1]
ax.fill_between(X_prop[:, 0], r_mean - 2*r_std, r_mean + 2*r_std, alpha=0.3, color="C1")
ax.plot(X_prop[:, 0], r_mean, "k-", lw=1.5)
ax.set(xlabel="x", ylabel="g(f(x))", title="After Nonlinear Transform (broadcast)")
plt.tight_layout()

## Summary

| Class | Role | Key Methods |
|:------|:-----|:------------|
| `RandomFunction[X, Y]` | Distribution over functions; `__call__` is the fundamental interface | `__call__(x) -> Distribution[Y]` |
| `ArrayRandomFunction` | Array-valued specialization with shape contract | `predict()`, `_parse_X()`, `_validate_X()` |
| `GaussianRandomFunction` | Gaussian predictive distributions | `predict_mean()`, `predict_variance()`, `predict_covariance()` |
| `LinearBasisFunction` | `f(x) = a + Φ(x)@w`, finite-dimensional | `sample()` returns callables |
| `LinearOutputTransform` | `f(x) = a + Φ @ g(x)`, linear projection | Wraps another `GaussianRandomFunction` |
| `EmulatorMixin` | Training-data interface (composable mixin) | `fit()`, `update()`, `training_inputs`, `training_responses` |

**Key design points:**

- Every random function is a `Distribution` — it lives in the same hierarchy
  as `Normal`, `MultivariateNormal`, etc.
- **Calling** a random function (not sampling it) is the primary interface:
  `rf(x)` returns a distribution over outputs.
- **Shape semantics** are controlled by `joint_inputs` / `joint_outputs` flags,
  which partition axes between batch and event dimensions.
- **Function sampling** is available for finite-dimensional models like
  `LinearBasisFunction` — `sample()` returns a callable.
- Outputs are standard ProbPipe distributions, so all existing features work:
  expectations, log-densities, provenance tracking, and WorkflowFunction
  broadcasting.